In [3]:
# ==========================================
# Phase 7.5 — Dataset Quality Analysis
# Cell 1 — Import & Environment
# ==========================================

from pathlib import Path
from datetime import datetime
from collections import Counter
import hashlib
import json
import random
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

# -------------------------------------------------
# Reproducibility
# -------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# -------------------------------------------------
# Display
# -------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

print("=" * 80)
print("Phase 7.5 — Dataset Quality Analysis")
print("=" * 80)
print("Environment initialized")
print(f"Random Seed : {RANDOM_SEED}")
print(f"Python      : {__import__('sys').version.split()[0]}")
print(f"Pandas      : {pd.__version__}")
print(f"NumPy       : {np.__version__}")
print("=" * 80)

Phase 7.5 — Dataset Quality Analysis
Environment initialized
Random Seed : 42
Python      : 3.10.11
Pandas      : 2.3.3
NumPy       : 1.26.4


In [4]:
# ==========================================
# Phase 7.5
# Cell 2 — Configuration & Paths
# ==========================================

print("=" * 80)
print("Phase 7.5 Configuration")
print("=" * 80)

# -------------------------------------------------
# Project Root
# -------------------------------------------------

PROJECT_DIR = Path(r"C:\LipReadingSSL").resolve()

OUTPUT_DIR = PROJECT_DIR / "output"

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}"
    )

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Output directory not found:\n{OUTPUT_DIR}"
    )

# -------------------------------------------------
# Phase 7 Output
# -------------------------------------------------

VIDEO_DIR = OUTPUT_DIR

DATASET_DIR = OUTPUT_DIR / "dataset"

# -------------------------------------------------
# Phase 7.5 Output
# -------------------------------------------------

ANALYSIS_DIR = DATASET_DIR / "analysis"

GRAPH_DIR = ANALYSIS_DIR / "graphs"

REPORT_DIR = ANALYSIS_DIR / "reports"

VOCAB_DIR = ANALYSIS_DIR / "vocabulary"

HISTORY_DIR = ANALYSIS_DIR / "history"

# -------------------------------------------------
# Create directories
# -------------------------------------------------

for folder in [
    DATASET_DIR,
    ANALYSIS_DIR,
    GRAPH_DIR,
    REPORT_DIR,
    VOCAB_DIR,
    HISTORY_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# -------------------------------------------------
# Dataset Files
# -------------------------------------------------

MASTER_FILE = DATASET_DIR / "dataset_master.csv"

LABEL_FILE = DATASET_DIR / "labels.csv"

SPLIT_FILE = DATASET_DIR / "split.csv"

TRAIN_FILE = DATASET_DIR / "train.csv"

VAL_FILE = DATASET_DIR / "val.csv"

TEST_FILE = DATASET_DIR / "test.csv"

# -------------------------------------------------
# QA Files
# -------------------------------------------------

FINGERPRINT_FILE = (
    ANALYSIS_DIR / "dataset_fingerprint.json"
)

HISTORY_FILE = (
    ANALYSIS_DIR / "dataset_history.csv"
)

SUMMARY_FILE = (
    ANALYSIS_DIR / "dataset_summary.csv"
)

QUALITY_FILE = (
    ANALYSIS_DIR / "quality_score.json"
)

REPORT_FILE = (
    REPORT_DIR / "dataset_qa_report.md"
)

# -------------------------------------------------
# Split Configuration
# -------------------------------------------------

TRAIN_RATIO = 0.80

VAL_RATIO = 0.10

TEST_RATIO = 0.10

SPLIT_MODE = "AUTO"

# -------------------------------------------------
# Dataset Version
# -------------------------------------------------

DATASET_VERSION = "7.5.0"

# -------------------------------------------------
# Print
# -------------------------------------------------

print(f"Project      : {PROJECT_DIR}")
print(f"Output       : {OUTPUT_DIR}")
print(f"Dataset      : {DATASET_DIR}")
print(f"Analysis     : {ANALYSIS_DIR}")
print(f"Graphs       : {GRAPH_DIR}")
print(f"Reports      : {REPORT_DIR}")
print()
print("Split Ratio")
print(f"Train : {TRAIN_RATIO:.0%}")
print(f"Val   : {VAL_RATIO:.0%}")
print(f"Test  : {TEST_RATIO:.0%}")
print()
print(f"Split Mode   : {SPLIT_MODE}")
print(f"Version      : {DATASET_VERSION}")

print("=" * 80)

Phase 7.5 Configuration
Project      : C:\LipReadingSSL
Output       : C:\LipReadingSSL\output
Dataset      : C:\LipReadingSSL\output\dataset
Analysis     : C:\LipReadingSSL\output\dataset\analysis
Graphs       : C:\LipReadingSSL\output\dataset\analysis\graphs
Reports      : C:\LipReadingSSL\output\dataset\analysis\reports

Split Ratio
Train : 80%
Val   : 10%
Test  : 10%

Split Mode   : AUTO
Version      : 7.5.0


In [5]:
# ==========================================
# Phase 7.5
# Cell 3 — Load Dataset Safely
# ==========================================

print("=" * 80)
print("Loading Phase 7 Dataset")
print("=" * 80)

# -------------------------------------------------
# Locate Master Dataset
# -------------------------------------------------

if MASTER_FILE.exists():

    dataset_file = MASTER_FILE

elif LABEL_FILE.exists():

    dataset_file = LABEL_FILE

else:

    # Search possible CSV files
    csv_candidates = sorted(
        DATASET_DIR.glob("*.csv")
    )

    if not csv_candidates:

        raise FileNotFoundError(
            "Cannot find dataset CSV.\n"
            f"Expected:\n{MASTER_FILE}\n"
            f"or:\n{LABEL_FILE}"
        )

    dataset_file = csv_candidates[0]

print(f"Dataset file : {dataset_file}")

# -------------------------------------------------
# Load
# -------------------------------------------------

dataset_df = pd.read_csv(
    dataset_file,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# Normalize column names
# -------------------------------------------------

dataset_df.columns = [

    str(col)
    .strip()
    .lower()
    .replace(" ", "_")

    for col in dataset_df.columns

]

# -------------------------------------------------
# Remove completely empty rows
# -------------------------------------------------

dataset_df = dataset_df.dropna(
    how="all"
).reset_index(
    drop=True
)

# -------------------------------------------------
# Required logical fields
# -------------------------------------------------

COLUMN_ALIASES = {

    "dataset_id": [
        "dataset_id",
        "clip_id",
        "sample_id",
        "id"
    ],

    "video_id": [
        "video_id",
        "video",
        "video_name"
    ],

    "speaker_id": [
        "speaker_id",
        "speaker",
        "speaker_name"
    ],

    "normalized_text": [
        "normalized_text",
        "text",
        "sentence",
        "transcript",
        "label"
    ],

    "num_frames": [
        "num_frames",
        "frames",
        "frame_count"
    ],

    "clip_path": [
        "clip_path",
        "video_path",
        "path"
    ],

    "frame_folder": [
        "frame_folder",
        "frames_path",
        "frame_dir"
    ]
}

# -------------------------------------------------
# Resolve aliases
# -------------------------------------------------

for canonical, aliases in COLUMN_ALIASES.items():

    if canonical in dataset_df.columns:
        continue

    for alias in aliases:

        if alias in dataset_df.columns:

            dataset_df[canonical] = (
                dataset_df[alias]
            )

            break

# -------------------------------------------------
# Safe defaults
# -------------------------------------------------

if "dataset_id" not in dataset_df.columns:

    dataset_df["dataset_id"] = [
        f"sample_{i:06d}"
        for i in range(len(dataset_df))
    ]

if "video_id" not in dataset_df.columns:

    dataset_df["video_id"] = "UNKNOWN_VIDEO"

if "speaker_id" not in dataset_df.columns:

    dataset_df["speaker_id"] = "UNKNOWN"

if "normalized_text" not in dataset_df.columns:

    dataset_df["normalized_text"] = ""

if "num_frames" not in dataset_df.columns:

    dataset_df["num_frames"] = np.nan

if "clip_path" not in dataset_df.columns:

    dataset_df["clip_path"] = ""

if "frame_folder" not in dataset_df.columns:

    dataset_df["frame_folder"] = ""

# -------------------------------------------------
# Clean values
# -------------------------------------------------

for col in [
    "dataset_id",
    "video_id",
    "speaker_id",
    "normalized_text",
    "clip_path",
    "frame_folder"
]:

    dataset_df[col] = (
        dataset_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

dataset_df["num_frames"] = pd.to_numeric(
    dataset_df["num_frames"],
    errors="coerce"
)

# -------------------------------------------------
# Basic information
# -------------------------------------------------

print()
print(f"Samples : {len(dataset_df):,}")
print(f"Columns : {len(dataset_df.columns)}")

print()
print("Columns detected:")

for col in dataset_df.columns:

    print(f" - {col}")

print()
print("Preview:")

display(
    dataset_df.head()
)

print("=" * 80)
print("Dataset loaded successfully")
print("=" * 80)

Loading Phase 7 Dataset
Dataset file : C:\LipReadingSSL\output\dataset\dataset_master.csv

Samples : 81,853
Columns : 20

Columns detected:
 - dataset_id
 - video_id
 - clip_id
 - clip_path
 - frame_folder
 - start_frame
 - end_frame
 - num_frames
 - clip_length
 - stride
 - speaker_id
 - sentence_id
 - transcript
 - normalized_text
 - language
 - verified
 - clip_exists
 - frame_exists
 - has_transcript
 - split

Preview:


,dataset_id,video_id,clip_id,clip_path,frame_folder,start_frame,end_frame,num_frames,clip_length,stride,speaker_id,sentence_id,transcript,normalized_text,language,verified,clip_exists,frame_exists,has_transcript,split
0,1,video001,clip_000001,video001\clip_npy\clip_000001.npy,video001\clips\clip_000001,0,25,29,29.0,1,UNKNOWN,NaN,NaN,,th,False,False,False,False,test
1,2,video001,clip_000002,video001\clip_npy\clip_000002.npy,video001\clips\clip_000002,0,26,29,29.0,1,UNKNOWN,NaN,NaN,,th,False,False,False,False,test
2,3,video001,clip_000003,video001\clip_npy\clip_000003.npy,video001\clips\clip_000003,1,27,29,29.0,1,UNKNOWN,NaN,NaN,,th,False,False,False,False,test
3,4,video001,clip_000004,video001\clip_npy\clip_000004.npy,video001\clips\clip_000004,2,28,29,29.0,1,UNKNOWN,NaN,NaN,,th,False,False,False,False,test
4,5,video001,clip_000005,video001\clip_npy\clip_000005.npy,video001\clips\clip_000005,3,29,29,29.0,1,UNKNOWN,NaN,NaN,,th,False,False,False,False,test


Dataset loaded successfully


In [6]:
# ==========================================
# Phase 7.5
# Cell 4 — Dataset Fingerprint + Stable Split Engine
# ==========================================

print("=" * 80)
print("Cell 4 — Dataset Fingerprint + Stable Split Engine")
print("=" * 80)

# -------------------------------------------------
# 1. Dataset Fingerprint
# -------------------------------------------------

def normalize_for_fingerprint(df):
    """
    Create a deterministic representation of the dataset.
    Used to detect whether the dataset content changed.
    """

    fingerprint_columns = [
        col for col in [
            "dataset_id",
            "video_id",
            "speaker_id",
            "normalized_text",
            "num_frames",
            "clip_path",
            "frame_folder"
        ]
        if col in df.columns
    ]

    temp = df[fingerprint_columns].copy()

    # Normalize values
    for col in temp.columns:
        temp[col] = (
            temp[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    # Stable ordering
    temp = temp.sort_values(
        by=fingerprint_columns,
        kind="mergesort"
    ).reset_index(drop=True)

    return temp


fingerprint_df = normalize_for_fingerprint(
    dataset_df
)

fingerprint_text = fingerprint_df.to_csv(
    index=False,
    lineterminator="\n"
)

sha256_hash = hashlib.sha256(
    fingerprint_text.encode("utf-8")
).hexdigest()

fingerprint_info = {
    "dataset_version": DATASET_VERSION,
    "sha256": sha256_hash,
    "samples": int(len(dataset_df)),
    "videos": int(dataset_df["video_id"].nunique()),
    "speakers": int(dataset_df["speaker_id"].nunique()),
    "generated_at": datetime.now().isoformat(
        timespec="seconds"
    )
}

with open(
    FINGERPRINT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        fingerprint_info,
        f,
        ensure_ascii=False,
        indent=4
    )

print()
print("Dataset Fingerprint")
print(f"SHA256  : {sha256_hash}")
print(f"Samples : {len(dataset_df):,}")
print(f"Videos  : {dataset_df['video_id'].nunique():,}")
print(f"Speakers: {dataset_df['speaker_id'].nunique():,}")

# -------------------------------------------------
# 2. Stable Hash
# -------------------------------------------------

def stable_hash(value, seed=RANDOM_SEED):
    """
    Deterministic hash.
    Unlike Python's built-in hash(),
    this produces the same result across sessions.
    """

    text = f"{seed}|{value}"

    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# -------------------------------------------------
# 3. Detect Dataset Structure
# -------------------------------------------------

valid_speakers = sorted(
    x for x in dataset_df["speaker_id"].unique()
    if x and x != "UNKNOWN"
)

valid_videos = sorted(
    x for x in dataset_df["video_id"].unique()
    if x and x != "UNKNOWN_VIDEO"
)

num_speakers = len(valid_speakers)
num_videos = len(valid_videos)

print()
print("Dataset Structure")
print(f"Valid Speakers : {num_speakers}")
print(f"Valid Videos   : {num_videos}")

# -------------------------------------------------
# 4. Split Strategy
# -------------------------------------------------

if SPLIT_MODE.upper() != "AUTO":

    selected_strategy = SPLIT_MODE.upper()

else:

    if num_speakers >= 3:

        selected_strategy = "SPEAKER"

    elif num_videos >= 3:

        selected_strategy = "VIDEO"

    elif num_videos == 2:

        selected_strategy = "TWO_VIDEO"

    elif num_videos == 1:

        selected_strategy = "SINGLE_VIDEO"

    else:

        selected_strategy = "CLIP"


print()
print("Selected Split Strategy :", selected_strategy)

# -------------------------------------------------
# 5. Generic Group Split
# -------------------------------------------------

def split_groups(groups):
    """
    Stable group splitting.

    1 group
        TRAIN

    2 groups
        TRAIN / TEST

    3+ groups
        TRAIN / VAL / TEST
    """

    groups = list(groups)

    groups = sorted(
        groups,
        key=lambda x: stable_hash(str(x))
    )

    n = len(groups)

    if n == 0:

        return [], [], []

    # ---------------------------------------------
    # One group
    # ---------------------------------------------

    if n == 1:

        return groups, [], []

    # ---------------------------------------------
    # Two groups
    # ---------------------------------------------

    if n == 2:

        return (
            groups[:1],
            [],
            groups[1:]
        )

    # ---------------------------------------------
    # Three or more
    # ---------------------------------------------

    test_n = max(
        1,
        round(n * TEST_RATIO)
    )

    val_n = max(
        1,
        round(n * VAL_RATIO)
    )

    # Ensure train remains
    if test_n + val_n >= n:

        test_n = 1
        val_n = 1

    train_n = n - val_n - test_n

    train_groups = groups[:train_n]

    val_groups = groups[
        train_n:
        train_n + val_n
    ]

    test_groups = groups[
        train_n + val_n:
    ]

    return (
        train_groups,
        val_groups,
        test_groups
    )


# -------------------------------------------------
# 6. Build Split
# -------------------------------------------------

dataset_df["split"] = "train"

if selected_strategy == "SPEAKER":

    train_groups, val_groups, test_groups = (
        split_groups(valid_speakers)
    )

    split_column = "speaker_id"

elif selected_strategy == "VIDEO":

    train_groups, val_groups, test_groups = (
        split_groups(valid_videos)
    )

    split_column = "video_id"

elif selected_strategy == "TWO_VIDEO":

    train_groups, val_groups, test_groups = (
        split_groups(valid_videos)
    )

    split_column = "video_id"

elif selected_strategy == "SINGLE_VIDEO":

    train_groups = valid_videos
    val_groups = []
    test_groups = []

    split_column = "video_id"

else:

    # Fallback:
    # split at sample level only when
    # no usable speaker/video information exists.

    dataset_df["_stable_order"] = (
        dataset_df["dataset_id"]
        .astype(str)
        .map(stable_hash)
    )

    dataset_df = dataset_df.sort_values(
        "_stable_order"
    ).reset_index(drop=True)

    n = len(dataset_df)

    if n >= 3:

        test_n = max(
            1,
            round(n * TEST_RATIO)
        )

        val_n = max(
            1,
            round(n * VAL_RATIO)
        )

        train_n = n - val_n - test_n

        dataset_df.loc[
            :train_n - 1,
            "split"
        ] = "train"

        dataset_df.loc[
            train_n:
            train_n + val_n - 1,
            "split"
        ] = "val"

        dataset_df.loc[
            train_n + val_n:,
            "split"
        ] = "test"

    elif n == 2:

        dataset_df.loc[
            0,
            "split"
        ] = "train"

        dataset_df.loc[
            1,
            "split"
        ] = "test"

    else:

        dataset_df["split"] = "train"

    split_column = "dataset_id"

    train_groups = []
    val_groups = []
    test_groups = []


# -------------------------------------------------
# 7. Apply Group Split
# -------------------------------------------------

if selected_strategy in [
    "SPEAKER",
    "VIDEO",
    "TWO_VIDEO"
]:

    dataset_df.loc[
        dataset_df[split_column].isin(train_groups),
        "split"
    ] = "train"

    dataset_df.loc[
        dataset_df[split_column].isin(val_groups),
        "split"
    ] = "val"

    dataset_df.loc[
        dataset_df[split_column].isin(test_groups),
        "split"
    ] = "test"


# -------------------------------------------------
# 8. Remove temporary column
# -------------------------------------------------

if "_stable_order" in dataset_df.columns:

    dataset_df = dataset_df.drop(
        columns=["_stable_order"]
    )


# -------------------------------------------------
# 9. Split Summary
# -------------------------------------------------

split_counts = (
    dataset_df["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"],
        fill_value=0
    )
)

print()
print("=" * 80)
print("Split Result")
print("=" * 80)

print(f"Train : {split_counts['train']:,}")
print(f"Val   : {split_counts['val']:,}")
print(f"Test  : {split_counts['test']:,}")

print()
print("Group Allocation")

if selected_strategy in [
    "SPEAKER",
    "VIDEO",
    "TWO_VIDEO"
]:

    print("Train:", train_groups)
    print("Val  :", val_groups)
    print("Test :", test_groups)

print("=" * 80)

Cell 4 — Dataset Fingerprint + Stable Split Engine

Dataset Fingerprint
SHA256  : 2c4324eb120e4bd8abe32edec7b12f7d725d7597fc6088fc96afa6dbcc7d4e2b
Samples : 81,853
Videos  : 2
Speakers: 1

Dataset Structure
Valid Speakers : 0
Valid Videos   : 2

Selected Split Strategy : TWO_VIDEO

Split Result
Train : 42,993
Val   : 0
Test  : 38,860

Group Allocation
Train: ['video001']
Val  : []
Test : ['video002']


In [7]:
# ==========================================
# Phase 7.5
# Cell 5 — Split Validation + Export
# ==========================================

print("=" * 80)
print("Cell 5 — Split Validation + Export")
print("=" * 80)

# -------------------------------------------------
# 1. Basic Split Validation
# -------------------------------------------------

VALID_SPLITS = {
    "train",
    "val",
    "test"
}

invalid_split_values = set(
    dataset_df["split"].dropna().unique()
) - VALID_SPLITS

if invalid_split_values:

    raise ValueError(
        f"Invalid split values detected: "
        f"{invalid_split_values}"
    )

# -------------------------------------------------
# 2. Split Counts
# -------------------------------------------------

split_counts = (
    dataset_df["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"],
        fill_value=0
    )
)

print()
print("Sample Distribution")

for split_name in [
    "train",
    "val",
    "test"
]:

    count = int(
        split_counts[split_name]
    )

    percentage = (
        count / len(dataset_df) * 100
        if len(dataset_df) > 0
        else 0
    )

    print(
        f"{split_name.upper():<6} "
        f"{count:>10,} "
        f"({percentage:6.2f}%)"
    )

# -------------------------------------------------
# 3. Group Leakage Check
# -------------------------------------------------

print()
print("=" * 80)
print("Leakage Check")
print("=" * 80)

leakage_errors = []

# ---------------------------------------------
# Speaker Leakage
# ---------------------------------------------

if "speaker_id" in dataset_df.columns:

    speaker_sets = {}

    for split_name in [
        "train",
        "val",
        "test"
    ]:

        speaker_sets[split_name] = set(
            dataset_df.loc[
                dataset_df["split"] == split_name,
                "speaker_id"
            ]
            .dropna()
            .astype(str)
        )

    train_val = (
        speaker_sets["train"]
        & speaker_sets["val"]
    )

    train_test = (
        speaker_sets["train"]
        & speaker_sets["test"]
    )

    val_test = (
        speaker_sets["val"]
        & speaker_sets["test"]
    )

    if selected_strategy == "SPEAKER":

        if train_val:
            leakage_errors.append(
                f"Speaker leakage TRAIN/VAL: {train_val}"
            )

        if train_test:
            leakage_errors.append(
                f"Speaker leakage TRAIN/TEST: {train_test}"
            )

        if val_test:
            leakage_errors.append(
                f"Speaker leakage VAL/TEST: {val_test}"
            )

# ---------------------------------------------
# Video Leakage
# ---------------------------------------------

if "video_id" in dataset_df.columns:

    video_sets = {}

    for split_name in [
        "train",
        "val",
        "test"
    ]:

        video_sets[split_name] = set(
            dataset_df.loc[
                dataset_df["split"] == split_name,
                "video_id"
            ]
            .dropna()
            .astype(str)
        )

    train_val_video = (
        video_sets["train"]
        & video_sets["val"]
    )

    train_test_video = (
        video_sets["train"]
        & video_sets["test"]
    )

    val_test_video = (
        video_sets["val"]
        & video_sets["test"]
    )

    if selected_strategy in [
        "VIDEO",
        "TWO_VIDEO"
    ]:

        if train_val_video:
            leakage_errors.append(
                f"Video leakage TRAIN/VAL: "
                f"{train_val_video}"
            )

        if train_test_video:
            leakage_errors.append(
                f"Video leakage TRAIN/TEST: "
                f"{train_test_video}"
            )

        if val_test_video:
            leakage_errors.append(
                f"Video leakage VAL/TEST: "
                f"{val_test_video}"
            )

# -------------------------------------------------
# 4. Duplicate Dataset ID Check
# -------------------------------------------------

duplicate_ids = (
    dataset_df["dataset_id"]
    .duplicated()
)

duplicate_count = int(
    duplicate_ids.sum()
)

if duplicate_count > 0:

    leakage_errors.append(
        f"Duplicate dataset_id: "
        f"{duplicate_count}"
    )

# -------------------------------------------------
# 5. Print Leakage Result
# -------------------------------------------------

if leakage_errors:

    print("FAIL")

    for error in leakage_errors:

        print(" -", error)

else:

    print("PASS")
    print("No split leakage detected.")

# -------------------------------------------------
# 6. Split Summary Table
# -------------------------------------------------

split_summary = (
    dataset_df
    .groupby("split")
    .agg(
        samples=("dataset_id", "count"),
        videos=("video_id", "nunique"),
        speakers=("speaker_id", "nunique"),
        sentences=("normalized_text", "nunique"),
        avg_frames=("num_frames", "mean"),
        median_frames=("num_frames", "median"),
    )
    .reindex(
        ["train", "val", "test"]
    )
    .reset_index()
)

split_summary["avg_frames"] = (
    split_summary["avg_frames"]
    .round(2)
)

split_summary["median_frames"] = (
    split_summary["median_frames"]
    .round(2)
)

display(split_summary)

# -------------------------------------------------
# 7. Export Master Dataset
# -------------------------------------------------

dataset_df.to_csv(
    MASTER_FILE,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 8. Export Split File
# -------------------------------------------------

split_columns = [
    "dataset_id",
    "video_id",
    "speaker_id",
    "normalized_text",
    "num_frames",
    "clip_path",
    "frame_folder",
    "split"
]

split_columns = [
    col for col in split_columns
    if col in dataset_df.columns
]

dataset_df[split_columns].to_csv(
    SPLIT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 9. Export Train / Val / Test
# -------------------------------------------------

train_df = dataset_df[
    dataset_df["split"] == "train"
].copy()

val_df = dataset_df[
    dataset_df["split"] == "val"
].copy()

test_df = dataset_df[
    dataset_df["split"] == "test"
].copy()

train_df.to_csv(
    TRAIN_FILE,
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    VAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    TEST_FILE,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 10. Export Split Summary
# -------------------------------------------------

split_summary.to_csv(
    ANALYSIS_DIR / "split_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 11. Final
# -------------------------------------------------

print()
print("=" * 80)
print("Split Files Saved")
print("=" * 80)

print(f"Master : {MASTER_FILE}")
print(f"Split  : {SPLIT_FILE}")
print(f"Train  : {TRAIN_FILE}")
print(f"Val    : {VAL_FILE}")
print(f"Test   : {TEST_FILE}")

print()
print("Leakage Status :",
      "PASS" if not leakage_errors else "FAIL")

print("=" * 80)

Cell 5 — Split Validation + Export

Sample Distribution
TRAIN      42,993 ( 52.52%)
VAL             0 (  0.00%)
TEST       38,860 ( 47.48%)

Leakage Check
PASS
No split leakage detected.


,split,samples,videos,speakers,sentences,avg_frames,median_frames
0,train,42993.0,1.0,1.0,1.0,29.0,29.0
1,val,NaN,NaN,NaN,NaN,NaN,NaN
2,test,38860.0,1.0,1.0,1.0,29.0,29.0



Split Files Saved
Master : C:\LipReadingSSL\output\dataset\dataset_master.csv
Split  : C:\LipReadingSSL\output\dataset\split.csv
Train  : C:\LipReadingSSL\output\dataset\train.csv
Val    : C:\LipReadingSSL\output\dataset\val.csv
Test   : C:\LipReadingSSL\output\dataset\test.csv

Leakage Status : PASS


In [8]:
# ==========================================
# Phase 7.5
# Cell 6 — Dataset Summary + Clip QA
# ==========================================

print("=" * 80)
print("Cell 6 — Dataset Summary + Clip QA")
print("=" * 80)

# -------------------------------------------------
# 1. Path Existence Helper
# -------------------------------------------------

def path_exists_safe(value):
    """
    Safely check whether a path exists.
    Empty / UNKNOWN values return False.
    """

    if value is None:
        return False

    value = str(value).strip()

    if not value:
        return False

    if value.lower() in [
        "nan",
        "none",
        "unknown",
        "unknown_video"
    ]:
        return False

    try:

        return Path(value).exists()

    except Exception:

        return False


# -------------------------------------------------
# 2. Frame Statistics
# -------------------------------------------------

frame_series = pd.to_numeric(
    dataset_df["num_frames"],
    errors="coerce"
)

valid_frames = frame_series[
    frame_series.notna()
    & (frame_series > 0)
]

# -------------------------------------------------
# 3. Clip Path QA
# -------------------------------------------------

if "clip_path" in dataset_df.columns:

    clip_exists = dataset_df[
        "clip_path"
    ].apply(
        path_exists_safe
    )

else:

    clip_exists = pd.Series(
        False,
        index=dataset_df.index
    )

# -------------------------------------------------
# 4. Frame Folder QA
# -------------------------------------------------

if "frame_folder" in dataset_df.columns:

    frame_exists = dataset_df[
        "frame_folder"
    ].apply(
        path_exists_safe
    )

else:

    frame_exists = pd.Series(
        False,
        index=dataset_df.index
    )

# -------------------------------------------------
# 5. Text QA
# -------------------------------------------------

text_series = (
    dataset_df["normalized_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

missing_text = (
    text_series == ""
)

# -------------------------------------------------
# 6. Speaker QA
# -------------------------------------------------

speaker_series = (
    dataset_df["speaker_id"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
)

unknown_speaker = (
    speaker_series.isin([
        "",
        "UNKNOWN",
        "unknown",
        "nan"
    ])
)

# -------------------------------------------------
# 7. Video QA
# -------------------------------------------------

video_series = (
    dataset_df["video_id"]
    .fillna("UNKNOWN_VIDEO")
    .astype(str)
    .str.strip()
)

unknown_video = (
    video_series.isin([
        "",
        "UNKNOWN_VIDEO",
        "unknown_video",
        "nan"
    ])
)

# -------------------------------------------------
# 8. Duplicate QA
# -------------------------------------------------

duplicate_dataset_id = (
    dataset_df["dataset_id"]
    .duplicated(keep=False)
)

# -------------------------------------------------
# 9. Invalid Frame QA
# -------------------------------------------------

invalid_frames = (
    frame_series.isna()
    | (frame_series <= 0)
)

# -------------------------------------------------
# 10. QA DataFrame
# -------------------------------------------------

qa_df = pd.DataFrame({

    "dataset_id":
        dataset_df["dataset_id"],

    "video_id":
        dataset_df["video_id"],

    "speaker_id":
        dataset_df["speaker_id"],

    "split":
        dataset_df["split"],

    "text_missing":
        missing_text,

    "speaker_unknown":
        unknown_speaker,

    "video_unknown":
        unknown_video,

    "invalid_frames":
        invalid_frames,

    "clip_exists":
        clip_exists,

    "frame_folder_exists":
        frame_exists,

    "duplicate_dataset_id":
        duplicate_dataset_id
})

# -------------------------------------------------
# 11. QA Summary
# -------------------------------------------------

qa_metrics = {

    "Total Samples":
        len(dataset_df),

    "Videos":
        dataset_df["video_id"].nunique(),

    "Speakers":
        dataset_df["speaker_id"].nunique(),

    "Sentences":
        dataset_df["normalized_text"].nunique(),

    "Train":
        int(
            (dataset_df["split"] == "train")
            .sum()
        ),

    "Validation":
        int(
            (dataset_df["split"] == "val")
            .sum()
        ),

    "Test":
        int(
            (dataset_df["split"] == "test")
            .sum()
        ),

    "Average Frames":
        round(
            valid_frames.mean(), 2
        ) if len(valid_frames) else np.nan,

    "Median Frames":
        round(
            valid_frames.median(), 2
        ) if len(valid_frames) else np.nan,

    "Min Frames":
        int(
            valid_frames.min()
        ) if len(valid_frames) else np.nan,

    "Max Frames":
        int(
            valid_frames.max()
        ) if len(valid_frames) else np.nan,

    "Clip Exists":
        int(clip_exists.sum()),

    "Frame Folder Exists":
        int(frame_exists.sum()),

    "Missing Text":
        int(missing_text.sum()),

    "Unknown Speaker":
        int(unknown_speaker.sum()),

    "Unknown Video":
        int(unknown_video.sum()),

    "Invalid Frames":
        int(invalid_frames.sum()),

    "Duplicate Dataset ID":
        int(duplicate_dataset_id.sum())
}

summary_df = pd.DataFrame(
    list(qa_metrics.items()),
    columns=["Metric", "Value"]
)

display(summary_df)

# -------------------------------------------------
# 12. Save Summary
# -------------------------------------------------

summary_df.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 13. Save Clip QA
# -------------------------------------------------

qa_df.to_csv(
    ANALYSIS_DIR / "clip_qa.csv",
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 14. Frame Distribution
# -------------------------------------------------

if len(valid_frames) > 0:

    print()
    print("Frame Statistics")

    print(
        f"Mean   : {valid_frames.mean():.2f}"
    )

    print(
        f"Median : {valid_frames.median():.2f}"
    )

    print(
        f"Min    : {valid_frames.min():.0f}"
    )

    print(
        f"Max    : {valid_frames.max():.0f}"
    )

# -------------------------------------------------
# 15. QA Status
# -------------------------------------------------

critical_errors = (
    int(missing_text.sum())
    + int(invalid_frames.sum())
    + int(duplicate_dataset_id.sum())
)

print()
print("=" * 80)

if critical_errors == 0:

    print("Clip QA : PASS")

else:

    print("Clip QA : WARNING")

    print(
        f"Critical issues detected: "
        f"{critical_errors}"
    )

print("=" * 80)

print()
print("Saved:")
print(f" - {SUMMARY_FILE}")
print(
    f" - {ANALYSIS_DIR / 'clip_qa.csv'}"
)

Cell 6 — Dataset Summary + Clip QA


,Metric,Value
0,Total Samples,81853.0
1,Videos,2.0
2,Speakers,1.0
3,Sentences,1.0
4,Train,42993.0
5,Validation,0.0
6,Test,38860.0
7,Average Frames,29.0
8,Median Frames,29.0
9,Min Frames,29.0



Frame Statistics
Mean   : 29.00
Median : 29.00
Min    : 29
Max    : 29

Clip QA : WARNING
Critical issues detected: 81853

Saved:
 - C:\LipReadingSSL\output\dataset\analysis\dataset_summary.csv
 - C:\LipReadingSSL\output\dataset\analysis\clip_qa.csv


In [9]:
# ==========================================
# Phase 7.5
# Cell 7 — Vocabulary QA + Vocabulary Export
# ==========================================

print("=" * 80)
print("Cell 7 — Vocabulary QA")
print("=" * 80)

# -------------------------------------------------
# 1. Prepare Text
# -------------------------------------------------

text_df = dataset_df.copy()

text_df["normalized_text"] = (
    text_df["normalized_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# -------------------------------------------------
# 2. Remove Empty Text
# -------------------------------------------------

valid_text_df = text_df[
    text_df["normalized_text"] != ""
].copy()

empty_text_count = len(text_df) - len(valid_text_df)

# -------------------------------------------------
# 3. Normalize Text for Vocabulary
# -------------------------------------------------

def normalize_sentence(text):

    text = str(text).strip()

    # Normalize multiple spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


valid_text_df["normalized_text"] = (
    valid_text_df["normalized_text"]
    .apply(normalize_sentence)
)

# -------------------------------------------------
# 4. Sentence Vocabulary
# -------------------------------------------------

sentence_vocab = sorted(
    valid_text_df["normalized_text"]
    .dropna()
    .unique()
    .tolist()
)

# -------------------------------------------------
# 5. Character Vocabulary
# -------------------------------------------------

all_text = " ".join(
    sentence_vocab
)

character_vocab = sorted(
    set(all_text)
)

# -------------------------------------------------
# 6. Word Vocabulary
# -------------------------------------------------

word_counter = Counter()

for sentence in sentence_vocab:

    words = sentence.split()

    word_counter.update(words)

word_vocab = sorted(
    word_counter.keys()
)

# -------------------------------------------------
# 7. Vocabulary Statistics
# -------------------------------------------------

vocabulary_summary = {

    "total_samples":
        int(len(dataset_df)),

    "samples_with_text":
        int(len(valid_text_df)),

    "empty_text":
        int(empty_text_count),

    "unique_sentences":
        int(len(sentence_vocab)),

    "unique_words":
        int(len(word_vocab)),

    "unique_characters":
        int(len(character_vocab))
}

vocabulary_summary_df = pd.DataFrame(
    list(vocabulary_summary.items()),
    columns=[
        "metric",
        "value"
    ]
)

display(
    vocabulary_summary_df
)

# -------------------------------------------------
# 8. Export Sentence Vocabulary
# -------------------------------------------------

sentence_vocab_df = pd.DataFrame({

    "sentence_id": [
        f"S{i:05d}"
        for i in range(
            1,
            len(sentence_vocab) + 1
        )
    ],

    "sentence": sentence_vocab

})

sentence_vocab_file = (
    VOCAB_DIR / "sentence_vocabulary.csv"
)

sentence_vocab_df.to_csv(
    sentence_vocab_file,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 9. Export Word Vocabulary
# -------------------------------------------------

word_vocab_df = pd.DataFrame({

    "word_id": [
        f"W{i:05d}"
        for i in range(
            1,
            len(word_vocab) + 1
        )
    ],

    "word": word_vocab,

    "frequency": [
        word_counter[word]
        for word in word_vocab
    ]

})

word_vocab_file = (
    VOCAB_DIR / "word_vocabulary.csv"
)

word_vocab_df.to_csv(
    word_vocab_file,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 10. Export Character Vocabulary
# -------------------------------------------------

character_vocab_df = pd.DataFrame({

    "char_id": [
        f"C{i:05d}"
        for i in range(
            1,
            len(character_vocab) + 1
        )
    ],

    "character": character_vocab

})

character_vocab_file = (
    VOCAB_DIR / "character_vocabulary.csv"
)

character_vocab_df.to_csv(
    character_vocab_file,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 11. Export JSON
# -------------------------------------------------

vocabulary_json = {

    "version": DATASET_VERSION,

    "generated_at":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "sentences":
        sentence_vocab,

    "words":
        word_vocab,

    "characters":
        character_vocab

}

vocabulary_json_file = (
    VOCAB_DIR / "vocabulary.json"
)

with open(
    vocabulary_json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vocabulary_json,
        f,
        ensure_ascii=False,
        indent=4
    )

# -------------------------------------------------
# 12. Save Summary
# -------------------------------------------------

vocabulary_summary_df.to_csv(
    VOCAB_DIR / "vocabulary_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 13. Final
# -------------------------------------------------

print()
print("=" * 80)
print("Vocabulary QA Completed")
print("=" * 80)

print(
    f"Unique Sentences : {len(sentence_vocab):,}"
)

print(
    f"Unique Words     : {len(word_vocab):,}"
)

print(
    f"Unique Characters: {len(character_vocab):,}"
)

print(
    f"Empty Text       : {empty_text_count:,}"
)

print()
print("Saved:")

print(
    f" - {sentence_vocab_file}"
)

print(
    f" - {word_vocab_file}"
)

print(
    f" - {character_vocab_file}"
)

print(
    f" - {vocabulary_json_file}"
)

print("=" * 80)

Cell 7 — Vocabulary QA


,metric,value
0,total_samples,81853
1,samples_with_text,0
2,empty_text,81853
3,unique_sentences,0
4,unique_words,0
5,unique_characters,0



Vocabulary QA Completed
Unique Sentences : 0
Unique Words     : 0
Unique Characters: 0
Empty Text       : 81,853

Saved:
 - C:\LipReadingSSL\output\dataset\analysis\vocabulary\sentence_vocabulary.csv
 - C:\LipReadingSSL\output\dataset\analysis\vocabulary\word_vocabulary.csv
 - C:\LipReadingSSL\output\dataset\analysis\vocabulary\character_vocabulary.csv
 - C:\LipReadingSSL\output\dataset\analysis\vocabulary\vocabulary.json


In [10]:
# ==========================================
# Phase 7.5
# Cell 8 — Speaker / Video QA
# ==========================================

print("=" * 80)
print("Cell 8 — Speaker / Video QA")
print("=" * 80)

# -------------------------------------------------
# 1. Speaker Summary
# -------------------------------------------------

speaker_summary = (
    dataset_df
    .groupby("speaker_id", dropna=False)
    .agg(
        samples=("dataset_id", "count"),
        videos=("video_id", "nunique"),
        sentences=("normalized_text", "nunique")
    )
    .reset_index()
    .sort_values(
        "samples",
        ascending=False
    )
)

# -------------------------------------------------
# 2. Video Summary
# -------------------------------------------------

video_summary = (
    dataset_df
    .groupby("video_id", dropna=False)
    .agg(
        samples=("dataset_id", "count"),
        speakers=("speaker_id", "nunique"),
        sentences=("normalized_text", "nunique")
    )
    .reset_index()
    .sort_values(
        "samples",
        ascending=False
    )
)

# -------------------------------------------------
# 3. Split × Speaker
# -------------------------------------------------

speaker_split_summary = pd.crosstab(
    dataset_df["speaker_id"],
    dataset_df["split"]
)

for split_name in [
    "train",
    "val",
    "test"
]:

    if split_name not in speaker_split_summary.columns:

        speaker_split_summary[
            split_name
        ] = 0

speaker_split_summary = (
    speaker_split_summary[
        ["train", "val", "test"]
    ]
    .reset_index()
)

# -------------------------------------------------
# 4. Split × Video
# -------------------------------------------------

video_split_summary = pd.crosstab(
    dataset_df["video_id"],
    dataset_df["split"]
)

for split_name in [
    "train",
    "val",
    "test"
]:

    if split_name not in video_split_summary.columns:

        video_split_summary[
            split_name
        ] = 0

video_split_summary = (
    video_split_summary[
        ["train", "val", "test"]
    ]
    .reset_index()
)

# -------------------------------------------------
# 5. Speaker Imbalance
# -------------------------------------------------

if len(speaker_summary) > 0:

    speaker_counts = (
        speaker_summary["samples"]
    )

    speaker_min = int(
        speaker_counts.min()
    )

    speaker_max = int(
        speaker_counts.max()
    )

    speaker_mean = float(
        speaker_counts.mean()
    )

    if speaker_min > 0:

        speaker_imbalance_ratio = (
            speaker_max / speaker_min
        )

    else:

        speaker_imbalance_ratio = np.inf

else:

    speaker_min = 0
    speaker_max = 0
    speaker_mean = 0
    speaker_imbalance_ratio = 0

# -------------------------------------------------
# 6. Video Imbalance
# -------------------------------------------------

if len(video_summary) > 0:

    video_counts = (
        video_summary["samples"]
    )

    video_min = int(
        video_counts.min()
    )

    video_max = int(
        video_counts.max()
    )

    video_mean = float(
        video_counts.mean()
    )

    if video_min > 0:

        video_imbalance_ratio = (
            video_max / video_min
        )

    else:

        video_imbalance_ratio = np.inf

else:

    video_min = 0
    video_max = 0
    video_mean = 0
    video_imbalance_ratio = 0

# -------------------------------------------------
# 7. Print Speaker Information
# -------------------------------------------------

print()
print("Speaker Distribution")
display(
    speaker_summary
)

print()
print("Video Distribution")
display(
    video_summary
)

print()
print("Speaker × Split")
display(
    speaker_split_summary
)

print()
print("Video × Split")
display(
    video_split_summary
)

# -------------------------------------------------
# 8. QA Status
# -------------------------------------------------

speaker_warning = (
    speaker_imbalance_ratio > 10
    and len(speaker_summary) > 1
)

video_warning = (
    video_imbalance_ratio > 10
    and len(video_summary) > 1
)

print()
print("=" * 80)

if speaker_warning:

    print(
        "WARNING — Strong speaker imbalance detected"
    )

    print(
        f"Max/Min ratio: "
        f"{speaker_imbalance_ratio:.2f}"
    )

else:

    print(
        "Speaker Distribution: PASS"
    )

if video_warning:

    print(
        "WARNING — Strong video imbalance detected"
    )

    print(
        f"Max/Min ratio: "
        f"{video_imbalance_ratio:.2f}"
    )

else:

    print(
        "Video Distribution: PASS"
    )

print("=" * 80)

# -------------------------------------------------
# 9. Save
# -------------------------------------------------

speaker_summary.to_csv(
    ANALYSIS_DIR / "speaker_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

video_summary.to_csv(
    ANALYSIS_DIR / "video_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

speaker_split_summary.to_csv(
    ANALYSIS_DIR / "speaker_split_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

video_split_summary.to_csv(
    ANALYSIS_DIR / "video_split_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("Speaker / Video QA files saved.")

Cell 8 — Speaker / Video QA

Speaker Distribution


,speaker_id,samples,videos,sentences
0,UNKNOWN,81853,2,1



Video Distribution


,video_id,samples,speakers,sentences
0,video001,42993,1,1
1,video002,38860,1,1



Speaker × Split


split,speaker_id,train,val,test
0,UNKNOWN,42993,0,38860



Video × Split


split,video_id,train,val,test
0,video001,42993,0,0
1,video002,0,0,38860



Speaker Distribution: PASS
Video Distribution: PASS

Speaker / Video QA files saved.


In [11]:
# ==========================================
# Phase 7.5
# Cell 9 — Duplicate + Outlier QA
# ==========================================

print("=" * 80)
print("Cell 9 — Duplicate + Outlier QA")
print("=" * 80)

# -------------------------------------------------
# 1. Duplicate Dataset IDs
# -------------------------------------------------

duplicate_id_df = dataset_df[
    dataset_df["dataset_id"]
    .duplicated(keep=False)
].copy()

# -------------------------------------------------
# 2. Duplicate Content
# -------------------------------------------------

duplicate_key_columns = [
    "video_id",
    "normalized_text",
    "num_frames"
]

duplicate_key_columns = [
    col
    for col in duplicate_key_columns
    if col in dataset_df.columns
]

if duplicate_key_columns:

    duplicate_content_mask = (
        dataset_df[
            duplicate_key_columns
        ]
        .astype(str)
        .duplicated(
            keep=False
        )
    )

    duplicate_content_df = (
        dataset_df[
            duplicate_content_mask
        ]
        .copy()
    )

else:

    duplicate_content_df = (
        pd.DataFrame()
    )

# -------------------------------------------------
# 3. Frame Outlier Detection
# -------------------------------------------------

frames = pd.to_numeric(
    dataset_df["num_frames"],
    errors="coerce"
)

valid_frame_mask = (
    frames.notna()
    & (frames > 0)
)

valid_frames = frames[
    valid_frame_mask
]

if len(valid_frames) >= 4:

    q1 = valid_frames.quantile(0.25)

    q3 = valid_frames.quantile(0.75)

    iqr = q3 - q1

    lower_bound = max(
        1,
        q1 - 1.5 * iqr
    )

    upper_bound = (
        q3 + 1.5 * iqr
    )

    frame_outlier_mask = (
        frames < lower_bound
    ) | (
        frames > upper_bound
    )

else:

    q1 = np.nan
    q3 = np.nan
    iqr = np.nan

    lower_bound = np.nan
    upper_bound = np.nan

    frame_outlier_mask = (
        pd.Series(
            False,
            index=dataset_df.index
        )
    )

frame_outlier_df = dataset_df[
    frame_outlier_mask
].copy()

# -------------------------------------------------
# 4. Extremely Short Clips
# -------------------------------------------------

SHORT_CLIP_THRESHOLD = 5

short_clip_df = dataset_df[
    frames.notna()
    & (frames < SHORT_CLIP_THRESHOLD)
].copy()

# -------------------------------------------------
# 5. Extremely Long Clips
# -------------------------------------------------

LONG_CLIP_THRESHOLD = 300

long_clip_df = dataset_df[
    frames.notna()
    & (frames > LONG_CLIP_THRESHOLD)
].copy()

# -------------------------------------------------
# 6. Missing Metadata
# -------------------------------------------------

metadata_columns = [
    "dataset_id",
    "video_id",
    "speaker_id",
    "normalized_text"
]

metadata_issue_mask = pd.Series(
    False,
    index=dataset_df.index
)

for col in metadata_columns:

    if col in dataset_df.columns:

        values = (
            dataset_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        metadata_issue_mask |= (
            values == ""
        )

metadata_issue_df = dataset_df[
    metadata_issue_mask
].copy()

# -------------------------------------------------
# 7. Outlier Summary
# -------------------------------------------------

outlier_summary = pd.DataFrame({

    "check": [
        "duplicate_dataset_id",
        "duplicate_content",
        "invalid_frames",
        "frame_outliers",
        "very_short_clips",
        "very_long_clips",
        "missing_metadata"
    ],

    "count": [
        len(duplicate_id_df),
        len(duplicate_content_df),
        int(
            (~valid_frame_mask).sum()
        ),
        len(frame_outlier_df),
        len(short_clip_df),
        len(long_clip_df),
        len(metadata_issue_df)
    ]

})

display(
    outlier_summary
)

# -------------------------------------------------
# 8. Save Detailed Reports
# -------------------------------------------------

duplicate_id_df.to_csv(
    ANALYSIS_DIR / "duplicate_dataset_ids.csv",
    index=False,
    encoding="utf-8-sig"
)

duplicate_content_df.to_csv(
    ANALYSIS_DIR / "duplicate_content.csv",
    index=False,
    encoding="utf-8-sig"
)

frame_outlier_df.to_csv(
    ANALYSIS_DIR / "frame_outliers.csv",
    index=False,
    encoding="utf-8-sig"
)

short_clip_df.to_csv(
    ANALYSIS_DIR / "short_clips.csv",
    index=False,
    encoding="utf-8-sig"
)

long_clip_df.to_csv(
    ANALYSIS_DIR / "long_clips.csv",
    index=False,
    encoding="utf-8-sig"
)

metadata_issue_df.to_csv(
    ANALYSIS_DIR / "metadata_issues.csv",
    index=False,
    encoding="utf-8-sig"
)

outlier_summary.to_csv(
    ANALYSIS_DIR / "outlier_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 9. Final Status
# -------------------------------------------------

critical_issues = (
    int(
        (~valid_frame_mask).sum()
    )
    +
    len(duplicate_id_df)
    +
    len(metadata_issue_df)
)

warnings_count = (
    len(frame_outlier_df)
    +
    len(short_clip_df)
    +
    len(long_clip_df)
    +
    len(duplicate_content_df)
)

print()
print("=" * 80)

if critical_issues == 0:

    print("Critical QA : PASS")

else:

    print(
        "Critical QA : FAIL"
    )

    print(
        f"Critical Issues: "
        f"{critical_issues:,}"
    )

if warnings_count == 0:

    print(
        "Outlier QA  : PASS"
    )

else:

    print(
        "Outlier QA  : WARNING"
    )

    print(
        f"Warnings: "
        f"{warnings_count:,}"
    )

print("=" * 80)

print()
print("Saved:")
print(" - duplicate_dataset_ids.csv")
print(" - duplicate_content.csv")
print(" - frame_outliers.csv")
print(" - short_clips.csv")
print(" - long_clips.csv")
print(" - metadata_issues.csv")
print(" - outlier_summary.csv")

Cell 9 — Duplicate + Outlier QA


,check,count
0,duplicate_dataset_id,0
1,duplicate_content,81853
2,invalid_frames,0
3,frame_outliers,0
4,very_short_clips,0
5,very_long_clips,0
6,missing_metadata,81853



Critical QA : FAIL
Critical Issues: 81,853
Outlier QA  : WARNING
Warnings: 81,853

Saved:
 - duplicate_dataset_ids.csv
 - duplicate_content.csv
 - frame_outliers.csv
 - short_clips.csv
 - long_clips.csv
 - metadata_issues.csv
 - outlier_summary.csv


In [13]:
# ==========================================
# Phase 7.5
# Cell 10 — Distribution & Balance Analysis
# ==========================================

print("=" * 80)
print("Cell 10 — Distribution & Balance Analysis")
print("=" * 80)

# -------------------------------------------------
# 1. Split Distribution
# -------------------------------------------------

split_distribution = (
    dataset_df["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"],
        fill_value=0
    )
    .reset_index()
)

split_distribution.columns = [
    "split",
    "samples"
]

split_distribution["percentage"] = (
    split_distribution["samples"]
    / max(len(dataset_df), 1)
    * 100
)

print()
print("Split Distribution")
display(split_distribution)

# -------------------------------------------------
# 2. Video Distribution
# -------------------------------------------------

video_distribution = (
    dataset_df
    .groupby("video_id")
    .size()
    .reset_index(
        name="samples"
    )
    .sort_values(
        "samples",
        ascending=False
    )
)

video_distribution["percentage"] = (
    video_distribution["samples"]
    / max(len(dataset_df), 1)
    * 100
)

print()
print("Video Distribution")
display(video_distribution)

# -------------------------------------------------
# 3. Speaker Distribution
# -------------------------------------------------

speaker_distribution = (
    dataset_df
    .groupby("speaker_id")
    .size()
    .reset_index(
        name="samples"
    )
    .sort_values(
        "samples",
        ascending=False
    )
)

speaker_distribution["percentage"] = (
    speaker_distribution["samples"]
    / max(len(dataset_df), 1)
    * 100
)

print()
print("Speaker Distribution")
display(speaker_distribution)

# -------------------------------------------------
# 4. Sentence Distribution
# -------------------------------------------------

sentence_distribution = (
    dataset_df
    .groupby("normalized_text")
    .size()
    .reset_index(
        name="samples"
    )
    .sort_values(
        "samples",
        ascending=False
    )
)

# -------------------------------------------------
# 5. Frame Distribution
# -------------------------------------------------

frame_values = pd.to_numeric(
    dataset_df["num_frames"],
    errors="coerce"
).dropna()

frame_values = frame_values[
    frame_values > 0
]

if len(frame_values) > 0:

    frame_statistics = pd.DataFrame({

        "metric": [
            "mean",
            "median",
            "std",
            "min",
            "max",
            "q1",
            "q3"
        ],

        "value": [

            frame_values.mean(),

            frame_values.median(),

            frame_values.std(),

            frame_values.min(),

            frame_values.max(),

            frame_values.quantile(0.25),

            frame_values.quantile(0.75)

        ]

    })

else:

    frame_statistics = pd.DataFrame({

        "metric": [
            "mean",
            "median",
            "std",
            "min",
            "max",
            "q1",
            "q3"
        ],

        "value": [
            np.nan
        ] * 7

    })

print()
print("Frame Statistics")
display(frame_statistics)

# -------------------------------------------------
# 6. Balance Metrics
# -------------------------------------------------

def calculate_balance_ratio(series):

    counts = (
        series
        .value_counts()
    )

    if len(counts) <= 1:
        return 1.0

    minimum = counts.min()
    maximum = counts.max()

    if minimum <= 0:
        return np.inf

    return float(
        maximum / minimum
    )


video_balance_ratio = (
    calculate_balance_ratio(
        dataset_df["video_id"]
    )
)

speaker_balance_ratio = (
    calculate_balance_ratio(
        dataset_df["speaker_id"]
    )
)

sentence_balance_ratio = (
    calculate_balance_ratio(
        dataset_df["normalized_text"]
    )
)

# -------------------------------------------------
# 7. Analysis Summary
# -------------------------------------------------

balance_summary = pd.DataFrame({

    "category": [
        "video",
        "speaker",
        "sentence"
    ],

    "unique_values": [
        dataset_df["video_id"].nunique(),
        dataset_df["speaker_id"].nunique(),
        dataset_df["normalized_text"].nunique()
    ],

    "max_min_ratio": [
        video_balance_ratio,
        speaker_balance_ratio,
        sentence_balance_ratio
    ]

})

print()
print("Balance Analysis")
display(balance_summary)

# -------------------------------------------------
# 8. Save Analysis
# -------------------------------------------------

split_distribution.to_csv(
    ANALYSIS_DIR / "distribution_split.csv",
    index=False,
    encoding="utf-8-sig"
)

video_distribution.to_csv(
    ANALYSIS_DIR / "distribution_video.csv",
    index=False,
    encoding="utf-8-sig"
)

speaker_distribution.to_csv(
    ANALYSIS_DIR / "distribution_speaker.csv",
    index=False,
    encoding="utf-8-sig"
)

sentence_distribution.to_csv(
    ANALYSIS_DIR / "distribution_sentence.csv",
    index=False,
    encoding="utf-8-sig"
)

frame_statistics.to_csv(
    ANALYSIS_DIR / "frame_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)

balance_summary.to_csv(
    ANALYSIS_DIR / "balance_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("Distribution analysis saved.")

Cell 10 — Distribution & Balance Analysis

Split Distribution


,split,samples,percentage
0,train,42993,52.524648
1,val,0,0.000000
2,test,38860,47.475352



Video Distribution


,video_id,samples,percentage
0,video001,42993,52.524648
1,video002,38860,47.475352



Speaker Distribution


...


Frame Statistics


,metric,value
0,mean,29.0
1,median,29.0
2,std,0.0
3,min,29.0
4,max,29.0
5,q1,29.0
6,q3,29.0



Balance Analysis


,category,unique_values,max_min_ratio
0,video,2,1.106356
1,speaker,1,1.000000
2,sentence,1,1.000000



Distribution analysis saved.


In [14]:
# ==========================================
# Phase 7.5
# Cell 11 — Dataset Visualization
# ==========================================

print("=" * 80)
print("Cell 11 — Dataset Visualization")
print("=" * 80)

# -------------------------------------------------
# Helper
# -------------------------------------------------

def save_plot(filename):

    path = GRAPH_DIR / filename

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {path}")


# -------------------------------------------------
# 1. Split Distribution
# -------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    split_distribution["split"],
    split_distribution["samples"]
)

plt.title(
    "Dataset Split Distribution"
)

plt.xlabel(
    "Split"
)

plt.ylabel(
    "Number of Samples"
)

save_plot(
    "01_split_distribution.png"
)


# -------------------------------------------------
# 2. Video Distribution
# -------------------------------------------------

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    video_distribution["video_id"].astype(str),
    video_distribution["samples"]
)

plt.title(
    "Samples per Video"
)

plt.xlabel(
    "Video"
)

plt.ylabel(
    "Samples"
)

plt.xticks(
    rotation=45,
    ha="right"
)

save_plot(
    "02_video_distribution.png"
)


# -------------------------------------------------
# 3. Speaker Distribution
# -------------------------------------------------

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    speaker_distribution["speaker_id"].astype(str),
    speaker_distribution["samples"]
)

plt.title(
    "Samples per Speaker"
)

plt.xlabel(
    "Speaker"
)

plt.ylabel(
    "Samples"
)

plt.xticks(
    rotation=45,
    ha="right"
)

save_plot(
    "03_speaker_distribution.png"
)


# -------------------------------------------------
# 4. Frame Distribution
# -------------------------------------------------

if len(frame_values) > 0:

    plt.figure(
        figsize=(10, 5)
    )

    plt.hist(
        frame_values,
        bins=min(
            30,
            max(5, len(frame_values) // 100)
        )
    )

    plt.title(
        "Clip Frame Distribution"
    )

    plt.xlabel(
        "Number of Frames"
    )

    plt.ylabel(
        "Number of Samples"
    )

    save_plot(
        "04_frame_distribution.png"
    )

else:

    print(
        "Frame visualization skipped: "
        "no valid frame values."
    )


# -------------------------------------------------
# 5. Split × Video
# -------------------------------------------------

split_video_table = pd.crosstab(
    dataset_df["video_id"],
    dataset_df["split"]
)

split_video_table = (
    split_video_table
    .reindex(
        columns=[
            "train",
            "val",
            "test"
        ],
        fill_value=0
    )
)

plt.figure(
    figsize=(10, 6)
)

split_video_table.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Video Samples by Split"
)

plt.xlabel(
    "Video"
)

plt.ylabel(
    "Samples"
)

plt.xticks(
    rotation=45,
    ha="right"
)

save_plot(
    "05_video_split_distribution.png"
)


# -------------------------------------------------
# 6. Sentence Frequency
# -------------------------------------------------

if len(sentence_distribution) > 0:

    top_sentences = (
        sentence_distribution
        .head(20)
        .sort_values(
            "samples"
        )
    )

    plt.figure(
        figsize=(10, 8)
    )

    plt.barh(
        range(len(top_sentences)),
        top_sentences["samples"]
    )

    plt.yticks(
        range(len(top_sentences)),
        top_sentences[
            "normalized_text"
        ]
    )

    plt.title(
        "Top Sentence Frequencies"
    )

    plt.xlabel(
        "Number of Samples"
    )

    save_plot(
        "06_sentence_frequency.png"
    )


# -------------------------------------------------
# 7. Visualization Index
# -------------------------------------------------

graph_files = sorted(
    GRAPH_DIR.glob("*.png")
)

visualization_index = pd.DataFrame({

    "file": [
        file.name
        for file in graph_files
    ],

    "path": [
        str(file)
        for file in graph_files
    ]

})

visualization_index.to_csv(
    ANALYSIS_DIR / "visualization_index.csv",
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# Final
# -------------------------------------------------

print()
print("=" * 80)
print(
    f"Generated {len(graph_files)} visualization(s)"
)
print("=" * 80)

Cell 11 — Dataset Visualization
Saved: C:\LipReadingSSL\output\dataset\analysis\graphs\01_split_distribution.png
Saved: C:\LipReadingSSL\output\dataset\analysis\graphs\02_video_distribution.png
Saved: C:\LipReadingSSL\output\dataset\analysis\graphs\03_speaker_distribution.png
Saved: C:\LipReadingSSL\output\dataset\analysis\graphs\04_frame_distribution.png
Saved: C:\LipReadingSSL\output\dataset\analysis\graphs\05_video_split_distribution.png
Saved: C:\LipReadingSSL\output\dataset\analysis\graphs\06_sentence_frequency.png

Generated 8 visualization(s)


<Figure size 1000x600 with 0 Axes>

In [15]:
# ==========================================
# Phase 7.5
# Cell 12 — QA Recommendation
# ==========================================

print("=" * 80)
print("Cell 12 — QA Recommendation")
print("=" * 80)

recommendations = []

# -------------------------------------------------
# Helper
# -------------------------------------------------

def add_result(
    category,
    status,
    message,
    recommendation
):

    recommendations.append({

        "category": category,

        "status": status,

        "message": message,

        "recommendation": recommendation

    })


# -------------------------------------------------
# 1. Dataset Size
# -------------------------------------------------

sample_count = len(dataset_df)

if sample_count == 0:

    add_result(
        "Dataset Size",
        "CRITICAL",
        "Dataset contains no samples.",
        "Add valid samples before training."
    )

elif sample_count < 100:

    add_result(
        "Dataset Size",
        "WARNING",
        f"Dataset contains only {sample_count:,} samples.",
        "Collect more data before serious model training."
    )

else:

    add_result(
        "Dataset Size",
        "PASS",
        f"Dataset contains {sample_count:,} samples.",
        "Dataset size is acceptable for further QA."
    )


# -------------------------------------------------
# 2. Train / Val / Test
# -------------------------------------------------

train_count = int(
    split_counts.get(
        "train",
        0
    )
)

val_count = int(
    split_counts.get(
        "val",
        0
    )
)

test_count = int(
    split_counts.get(
        "test",
        0
    )
)

if train_count == 0:

    add_result(
        "Train Split",
        "CRITICAL",
        "Training split is empty.",
        "Create a valid training split."
    )

else:

    add_result(
        "Train Split",
        "PASS",
        f"Training samples: {train_count:,}.",
        "Training split is available."
    )


if val_count == 0:

    add_result(
        "Validation Split",
        "WARNING",
        "Validation split is empty.",
        "Add more independent groups such as speakers/videos "
        "when the dataset becomes larger."
    )

else:

    add_result(
        "Validation Split",
        "PASS",
        f"Validation samples: {val_count:,}.",
        "Validation split is available."
    )


if test_count == 0:

    add_result(
        "Test Split",
        "WARNING",
        "Test split is empty.",
        "Add at least one independent group for final evaluation."
    )

else:

    add_result(
        "Test Split",
        "PASS",
        f"Test samples: {test_count:,}.",
        "Test split is available."
    )


# -------------------------------------------------
# 3. Missing Text
# -------------------------------------------------

missing_text_count = int(
    (
        dataset_df["normalized_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    ).sum()
)

if missing_text_count > 0:

    add_result(
        "Text Metadata",
        "CRITICAL",
        f"{missing_text_count:,} samples have missing text.",
        "Fix transcript/label metadata."
    )

else:

    add_result(
        "Text Metadata",
        "PASS",
        "All samples contain text.",
        "Text metadata is complete."
    )


# -------------------------------------------------
# 4. Frame Quality
# -------------------------------------------------

invalid_frame_count = int(
    (
        pd.to_numeric(
            dataset_df["num_frames"],
            errors="coerce"
        ).isna()
        |
        (
            pd.to_numeric(
                dataset_df["num_frames"],
                errors="coerce"
            ) <= 0
        )
    ).sum()
)

if invalid_frame_count > 0:

    add_result(
        "Frame Metadata",
        "CRITICAL",
        f"{invalid_frame_count:,} samples have invalid frame counts.",
        "Re-run clip generation or repair frame metadata."
    )

else:

    add_result(
        "Frame Metadata",
        "PASS",
        "Frame metadata is valid.",
        "No immediate action required."
    )


# -------------------------------------------------
# 5. Duplicate IDs
# -------------------------------------------------

duplicate_count = int(
    dataset_df["dataset_id"]
    .duplicated()
    .sum()
)

if duplicate_count > 0:

    add_result(
        "Dataset IDs",
        "CRITICAL",
        f"{duplicate_count:,} duplicate dataset IDs detected.",
        "Regenerate unique dataset IDs."
    )

else:

    add_result(
        "Dataset IDs",
        "PASS",
        "Dataset IDs are unique.",
        "No immediate action required."
    )


# -------------------------------------------------
# 6. Speaker Count
# -------------------------------------------------

speaker_count = (
    dataset_df["speaker_id"]
    .nunique()
)

if speaker_count < 3:

    add_result(
        "Speaker Diversity",
        "WARNING",
        f"Only {speaker_count} speaker(s) detected.",
        "Collect more speakers to reduce speaker-specific bias."
    )

else:

    add_result(
        "Speaker Diversity",
        "PASS",
        f"{speaker_count} speakers detected.",
        "Speaker diversity is suitable for further analysis."
    )


# -------------------------------------------------
# 7. Video Count
# -------------------------------------------------

video_count = (
    dataset_df["video_id"]
    .nunique()
)

if video_count < 3:

    add_result(
        "Video Diversity",
        "WARNING",
        f"Only {video_count} video(s) detected.",
        "Add more independent videos/sessions."
    )

else:

    add_result(
        "Video Diversity",
        "PASS",
        f"{video_count} videos detected.",
        "Video diversity is acceptable."
    )


# -------------------------------------------------
# 8. Clip Existence
# -------------------------------------------------

if "clip_exists" in qa_df.columns:

    missing_clips = int(
        (~qa_df["clip_exists"]).sum()
    )

else:

    missing_clips = 0

if missing_clips > 0:

    add_result(
        "Clip Files",
        "WARNING",
        f"{missing_clips:,} clip paths could not be verified.",
        "Check clip paths and generated files."
    )

else:

    add_result(
        "Clip Files",
        "PASS",
        "All referenced clip paths are valid.",
        "No immediate action required."
    )


# -------------------------------------------------
# 9. Final Recommendation Table
# -------------------------------------------------

recommendation_df = pd.DataFrame(
    recommendations
)

display(
    recommendation_df
)

# -------------------------------------------------
# 10. Status Counts
# -------------------------------------------------

status_counts = (
    recommendation_df["status"]
    .value_counts()
)

critical_count = int(
    status_counts.get(
        "CRITICAL",
        0
    )
)

warning_count = int(
    status_counts.get(
        "WARNING",
        0
    )
)

pass_count = int(
    status_counts.get(
        "PASS",
        0
    )
)

# -------------------------------------------------
# 11. Overall Recommendation
# -------------------------------------------------

if critical_count > 0:

    overall_recommendation = "CRITICAL"

elif warning_count > 0:

    overall_recommendation = "WARNING"

else:

    overall_recommendation = "PASS"


print()
print("=" * 80)
print("Overall Recommendation")
print("=" * 80)

print(
    f"Status   : {overall_recommendation}"
)

print(
    f"PASS     : {pass_count}"
)

print(
    f"WARNING  : {warning_count}"
)

print(
    f"CRITICAL : {critical_count}"
)

print("=" * 80)

# -------------------------------------------------
# 12. Save
# -------------------------------------------------

recommendation_df.to_csv(
    ANALYSIS_DIR / "qa_recommendations.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    QUALITY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        {
            "overall_status":
                overall_recommendation,

            "pass":
                pass_count,

            "warning":
                warning_count,

            "critical":
                critical_count,

            "generated_at":
                datetime.now().isoformat(
                    timespec="seconds"
                )

        },

        f,

        ensure_ascii=False,

        indent=4

    )

print()
print(
    f"Saved: {ANALYSIS_DIR / 'qa_recommendations.csv'}"
)

print(
    f"Saved: {QUALITY_FILE}"
)

Cell 12 — QA Recommendation


,category,status,message,recommendation
0,Dataset Size,PASS,"Dataset contains 81,853 samples.",Dataset size is acceptable for further QA.
1,Train Split,PASS,"Training samples: 42,993.",Training split is available.
2,Validation Split,WARNING,Validation split is empty.,Add more independent groups such as speakers/videos when the dataset becomes larger.
3,Test Split,PASS,"Test samples: 38,860.",Test split is available.
4,Text Metadata,CRITICAL,"81,853 samples have missing text.",Fix transcript/label metadata.
5,Frame Metadata,PASS,Frame metadata is valid.,No immediate action required.
6,Dataset IDs,PASS,Dataset IDs are unique.,No immediate action required.
7,Speaker Diversity,WARNING,Only 1 speaker(s) detected.,Collect more speakers to reduce speaker-specific bias.
8,Video Diversity,WARNING,Only 2 video(s) detected.,Add more independent videos/sessions.
9,Clip Files,WARNING,"81,853 clip paths could not be verified.",Check clip paths and generated files.



Overall Recommendation
Status   : CRITICAL
PASS     : 5
CRITICAL : 1

Saved: C:\LipReadingSSL\output\dataset\analysis\qa_recommendations.csv
Saved: C:\LipReadingSSL\output\dataset\analysis\quality_score.json


In [16]:
# ==========================================
# Phase 7.5
# Cell 13 — Dataset History
# ==========================================

print("=" * 80)
print("Cell 13 — Dataset History")
print("=" * 80)

# -------------------------------------------------
# 1. History File
# -------------------------------------------------

HISTORY_FILE = (
    ANALYSIS_DIR / "dataset_history.csv"
)

HISTORY_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

# -------------------------------------------------
# 2. Safe Fingerprint
# -------------------------------------------------

current_fingerprint = {}

if "fingerprint" in globals():

    if isinstance(
        fingerprint,
        dict
    ):

        current_fingerprint = fingerprint.copy()

elif "dataset_fingerprint" in globals():

    if isinstance(
        dataset_fingerprint,
        dict
    ):

        current_fingerprint = (
            dataset_fingerprint.copy()
        )

# -------------------------------------------------
# 3. Extract Fingerprint
# -------------------------------------------------

fingerprint_sha256 = str(
    current_fingerprint.get(
        "sha256",
        ""
    )
)

fingerprint_short = str(
    current_fingerprint.get(
        "short",
        fingerprint_sha256[:12]
    )
)

# -------------------------------------------------
# 4. Dataset Statistics
# -------------------------------------------------

history_record = {

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "dataset_version":
        str(
            globals().get(
                "DATASET_VERSION",
                "unknown"
            )
        ),

    "fingerprint":
        fingerprint_sha256,

    "fingerprint_short":
        fingerprint_short,

    "samples":
        int(len(dataset_df)),

    "videos":
        int(
            dataset_df["video_id"]
            .nunique()
        ),

    "speakers":
        int(
            dataset_df["speaker_id"]
            .nunique()
        ),

    "sentences":
        int(
            dataset_df["normalized_text"]
            .nunique()
        ),

    "train":
        int(
            dataset_df[
                dataset_df["split"] == "train"
            ].shape[0]
        ),

    "val":
        int(
            dataset_df[
                dataset_df["split"] == "val"
            ].shape[0]
        ),

    "test":
        int(
            dataset_df[
                dataset_df["split"] == "test"
            ].shape[0]
        ),

    "qa_status":
        str(
            globals().get(
                "overall_recommendation",
                "UNKNOWN"
            )
        )

}

# -------------------------------------------------
# 5. Load Existing History Safely
# -------------------------------------------------

if HISTORY_FILE.exists():

    try:

        history_df = pd.read_csv(
            HISTORY_FILE
        )

    except Exception:

        history_df = pd.DataFrame()

else:

    history_df = pd.DataFrame()

# -------------------------------------------------
# 6. Schema Protection
# -------------------------------------------------

required_history_columns = [
    "timestamp",
    "dataset_version",
    "fingerprint",
    "fingerprint_short",
    "samples",
    "videos",
    "speakers",
    "sentences",
    "train",
    "val",
    "test",
    "qa_status"
]

if history_df.empty:

    history_df = pd.DataFrame(
        columns=required_history_columns
    )

else:

    # Add missing columns
    for column in required_history_columns:

        if column not in history_df.columns:

            history_df[column] = np.nan

    # Keep only known columns
    history_df = history_df[
        required_history_columns
    ]

# -------------------------------------------------
# 7. Check Duplicate Fingerprint
# -------------------------------------------------

fingerprint_exists = False

if fingerprint_sha256:

    existing_fingerprints = (
        history_df["fingerprint"]
        .fillna("")
        .astype(str)
        .values
    )

    fingerprint_exists = (
        fingerprint_sha256
        in existing_fingerprints
    )

# -------------------------------------------------
# 8. Append New Dataset Version
# -------------------------------------------------

if not fingerprint_exists:

    history_df = pd.concat(

        [
            history_df,

            pd.DataFrame(
                [history_record]
            )

        ],

        ignore_index=True

    )

    history_added = True

else:

    history_added = False

# -------------------------------------------------
# 9. Sort History
# -------------------------------------------------

if not history_df.empty:

    history_df = (
        history_df
        .sort_values(
            "timestamp"
        )
        .reset_index(
            drop=True
        )
    )

# -------------------------------------------------
# 10. Save
# -------------------------------------------------

history_df.to_csv(
    HISTORY_FILE,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 11. Display
# -------------------------------------------------

print()

if history_added:

    print(
        "New dataset version recorded."
    )

else:

    print(
        "Dataset fingerprint already exists."
    )

print()

print(
    f"History Records : {len(history_df):,}"
)

print(
    f"Current Samples : {len(dataset_df):,}"
)

print(
    f"Fingerprint     : "
    f"{fingerprint_short or 'N/A'}"
)

print()

display(
    history_df.tail(10)
)

print()
print(
    f"Saved: {HISTORY_FILE}"
)

print("=" * 80)

Cell 13 — Dataset History

New dataset version recorded.

History Records : 2
Current Samples : 81,853
Fingerprint     : N/A



,timestamp,dataset_version,fingerprint,fingerprint_short,samples,videos,speakers,sentences,train,val,test,qa_status
0,2026-08-16T14:35:14,7.5.0,,,81853,2,1,1.0,42993.0,0.0,38860.0,CRITICAL
1,NaN,NaN,NaN,NaN,81853,2,1,NaN,NaN,NaN,NaN,NaN



Saved: C:\LipReadingSSL\output\dataset\analysis\dataset_history.csv


In [17]:
# ==========================================
# Phase 7.5
# Cell 14 — Export Final QA Report
# ==========================================

print("=" * 80)
print("Cell 14 — Export Final QA Report")
print("=" * 80)

# -------------------------------------------------
# 1. Report Directory
# -------------------------------------------------

REPORT_DIR = (
    ANALYSIS_DIR / "final_report"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -------------------------------------------------
# 2. Current Dataset Information
# -------------------------------------------------

current_samples = int(
    len(dataset_df)
)

current_videos = int(
    dataset_df["video_id"]
    .nunique()
)

current_speakers = int(
    dataset_df["speaker_id"]
    .nunique()
)

current_sentences = int(
    dataset_df["normalized_text"]
    .nunique()
)

current_train = int(
    (
        dataset_df["split"] == "train"
    ).sum()
)

current_val = int(
    (
        dataset_df["split"] == "val"
    ).sum()
)

current_test = int(
    (
        dataset_df["split"] == "test"
    ).sum()
)

# -------------------------------------------------
# 3. Percentages
# -------------------------------------------------

def percentage(
    value,
    total
):

    if total <= 0:

        return 0.0

    return (
        float(value)
        / float(total)
        * 100
    )


train_pct = percentage(
    current_train,
    current_samples
)

val_pct = percentage(
    current_val,
    current_samples
)

test_pct = percentage(
    current_test,
    current_samples
)

# -------------------------------------------------
# 4. QA Status
# -------------------------------------------------

qa_status = str(
    globals().get(
        "overall_recommendation",
        "UNKNOWN"
    )
)

pass_count = int(
    globals().get(
        "pass_count",
        0
    )
)

warning_count = int(
    globals().get(
        "warning_count",
        0
    )
)

critical_count = int(
    globals().get(
        "critical_count",
        0
    )
)

# -------------------------------------------------
# 5. Report Summary
# -------------------------------------------------

report_summary = pd.DataFrame({

    "metric": [

        "Dataset Samples",

        "Videos",

        "Speakers",

        "Unique Sentences",

        "Train Samples",

        "Validation Samples",

        "Test Samples",

        "Train Percentage",

        "Validation Percentage",

        "Test Percentage",

        "QA PASS",

        "QA WARNING",

        "QA CRITICAL",

        "Overall Status"

    ],

    "value": [

        current_samples,

        current_videos,

        current_speakers,

        current_sentences,

        current_train,

        current_val,

        current_test,

        round(train_pct, 2),

        round(val_pct, 2),

        round(test_pct, 2),

        pass_count,

        warning_count,

        critical_count,

        qa_status

    ]

})

# -------------------------------------------------
# 6. Export CSV
# -------------------------------------------------

report_summary_file = (
    REPORT_DIR / "final_qa_summary.csv"
)

report_summary.to_csv(
    report_summary_file,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------
# 7. Export Recommendation
# -------------------------------------------------

if "recommendation_df" in globals():

    recommendation_file = (
        REPORT_DIR /
        "qa_recommendations.csv"
    )

    recommendation_df.to_csv(
        recommendation_file,
        index=False,
        encoding="utf-8-sig"
    )

# -------------------------------------------------
# 8. Export Balance
# -------------------------------------------------

if "balance_summary" in globals():

    balance_file = (
        REPORT_DIR /
        "balance_summary.csv"
    )

    balance_summary.to_csv(
        balance_file,
        index=False,
        encoding="utf-8-sig"
    )

# -------------------------------------------------
# 9. Export Outlier Summary
# -------------------------------------------------

if "outlier_summary" in globals():

    outlier_file = (
        REPORT_DIR /
        "outlier_summary.csv"
    )

    outlier_summary.to_csv(
        outlier_file,
        index=False,
        encoding="utf-8-sig"
    )

# -------------------------------------------------
# 10. Generate TXT Report
# -------------------------------------------------

report_txt = (
    REPORT_DIR /
    "dataset_qa_report.txt"
)

with open(
    report_txt,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "=" * 80 + "\n"
    )

    f.write(
        "PHASE 7.5 — DATASET QA REPORT\n"
    )

    f.write(
        "=" * 80 + "\n\n"
    )

    f.write(
        f"Generated : "
        f"{datetime.now().isoformat(timespec='seconds')}\n"
    )

    f.write(
        f"Status    : {qa_status}\n\n"
    )

    f.write(
        "DATASET\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        f"Samples   : {current_samples:,}\n"
    )

    f.write(
        f"Videos    : {current_videos:,}\n"
    )

    f.write(
        f"Speakers  : {current_speakers:,}\n"
    )

    f.write(
        f"Sentences : {current_sentences:,}\n\n"
    )

    f.write(
        "SPLIT\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        f"Train : {current_train:,} "
        f"({train_pct:.2f}%)\n"
    )

    f.write(
        f"Val   : {current_val:,} "
        f"({val_pct:.2f}%)\n"
    )

    f.write(
        f"Test  : {current_test:,} "
        f"({test_pct:.2f}%)\n\n"
    )

    f.write(
        "QA RESULT\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        f"PASS     : {pass_count}\n"
    )

    f.write(
        f"WARNING  : {warning_count}\n"
    )

    f.write(
        f"CRITICAL : {critical_count}\n"
    )

    f.write(
        f"STATUS   : {qa_status}\n"
    )

# -------------------------------------------------
# 11. Display
# -------------------------------------------------

print()
print("Final QA Summary")
print()

display(
    report_summary
)

print()
print("=" * 80)
print("Final QA Report Exported")
print("=" * 80)

print(
    f"CSV : {report_summary_file}"
)

print(
    f"TXT : {report_txt}"
)

print(
    f"DIR : {REPORT_DIR}"
)

print("=" * 80)

Cell 14 — Export Final QA Report

Final QA Summary



,metric,value
0,Dataset Samples,81853
1,Videos,2
2,Speakers,1
3,Unique Sentences,1
4,Train Samples,42993
5,Validation Samples,0
6,Test Samples,38860
7,Train Percentage,52.52
8,Validation Percentage,0.0
9,Test Percentage,47.48



Final QA Report Exported
CSV : C:\LipReadingSSL\output\dataset\analysis\final_report\final_qa_summary.csv
TXT : C:\LipReadingSSL\output\dataset\analysis\final_report\dataset_qa_report.txt
DIR : C:\LipReadingSSL\output\dataset\analysis\final_report


In [18]:
# ==========================================
# Phase 7.5
# Cell 15 — Final PASS / FAIL
# ==========================================

print("=" * 80)
print("Cell 15 — Final PASS / FAIL")
print("=" * 80)

# -------------------------------------------------
# 1. Collect Final Checks
# -------------------------------------------------

final_checks = []

def final_check(
    name,
    condition,
    severity,
    message
):

    final_checks.append({

        "check": name,

        "status":
            "PASS"
            if condition
            else severity,

        "message": message

    })


# -------------------------------------------------
# 2. Dataset Exists
# -------------------------------------------------

final_check(

    "Dataset Not Empty",

    len(dataset_df) > 0,

    "CRITICAL",

    f"{len(dataset_df):,} samples detected."

)


# -------------------------------------------------
# 3. Dataset ID
# -------------------------------------------------

duplicate_ids = int(
    dataset_df["dataset_id"]
    .duplicated()
    .sum()
)

final_check(

    "Unique Dataset IDs",

    duplicate_ids == 0,

    "CRITICAL",

    f"{duplicate_ids:,} duplicate IDs."

)


# -------------------------------------------------
# 4. Text
# -------------------------------------------------

missing_text = int(

    (
        dataset_df["normalized_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    ).sum()

)

final_check(

    "Text Metadata",

    missing_text == 0,

    "CRITICAL",

    f"{missing_text:,} samples without text."

)


# -------------------------------------------------
# 5. Frames
# -------------------------------------------------

frame_series = pd.to_numeric(

    dataset_df["num_frames"],

    errors="coerce"

)

invalid_frames = int(

    (
        frame_series.isna()
        |
        (frame_series <= 0)
    ).sum()

)

final_check(

    "Valid Frame Metadata",

    invalid_frames == 0,

    "CRITICAL",

    f"{invalid_frames:,} invalid frame records."

)


# -------------------------------------------------
# 6. Train
# -------------------------------------------------

final_check(

    "Train Split",

    current_train > 0,

    "CRITICAL",

    f"{current_train:,} training samples."

)


# -------------------------------------------------
# 7. Validation
# -------------------------------------------------

final_check(

    "Validation Split",

    current_val > 0,

    "WARNING",

    f"{current_val:,} validation samples."

)


# -------------------------------------------------
# 8. Test
# -------------------------------------------------

final_check(

    "Test Split",

    current_test > 0,

    "WARNING",

    f"{current_test:,} test samples."

)


# -------------------------------------------------
# 9. Speaker Diversity
# -------------------------------------------------

final_check(

    "Speaker Diversity",

    current_speakers >= 3,

    "WARNING",

    f"{current_speakers:,} speaker(s)."

)


# -------------------------------------------------
# 10. Video Diversity
# -------------------------------------------------

final_check(

    "Video Diversity",

    current_videos >= 3,

    "WARNING",

    f"{current_videos:,} video(s)."

)


# -------------------------------------------------
# 11. Duplicate Content
# -------------------------------------------------

duplicate_content_count = 0

if (
    "duplicate_content_df"
    in globals()
):

    duplicate_content_count = (
        len(
            duplicate_content_df
        )
    )

final_check(

    "Duplicate Content",

    duplicate_content_count == 0,

    "WARNING",

    f"{duplicate_content_count:,} duplicate-content records."

)


# -------------------------------------------------
# 12. Build Final Table
# -------------------------------------------------

final_check_df = pd.DataFrame(
    final_checks
)

display(
    final_check_df
)

# -------------------------------------------------
# 13. Count Status
# -------------------------------------------------

critical_failures = int(

    (
        final_check_df["status"]
        == "CRITICAL"
    ).sum()

)

warnings = int(

    (
        final_check_df["status"]
        == "WARNING"
    ).sum()

)

passed = int(

    (
        final_check_df["status"]
        == "PASS"
    ).sum()

)

# -------------------------------------------------
# 14. Final Decision
# -------------------------------------------------

if critical_failures > 0:

    FINAL_STATUS = "FAIL"

    FINAL_DECISION = (
        "Dataset is NOT ready for training."
    )

elif warnings > 0:

    FINAL_STATUS = "PASS_WITH_WARNING"

    FINAL_DECISION = (
        "Dataset can continue, "
        "but warnings should be reviewed."
    )

else:

    FINAL_STATUS = "PASS"

    FINAL_DECISION = (
        "Dataset passed all QA checks."
    )

# -------------------------------------------------
# 15. Save Final Result
# -------------------------------------------------

final_result = {

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "dataset_version":
        str(
            globals().get(
                "DATASET_VERSION",
                "unknown"
            )
        ),

    "samples":
        current_samples,

    "videos":
        current_videos,

    "speakers":
        current_speakers,

    "sentences":
        current_sentences,

    "train":
        current_train,

    "val":
        current_val,

    "test":
        current_test,

    "passed_checks":
        passed,

    "warnings":
        warnings,

    "critical_failures":
        critical_failures,

    "status":
        FINAL_STATUS,

    "decision":
        FINAL_DECISION

}

final_result_file = (
    REPORT_DIR /
    "final_result.json"
)

with open(
    final_result_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_result,
        f,
        ensure_ascii=False,
        indent=4
    )

final_check_df.to_csv(

    REPORT_DIR /
    "final_checks.csv",

    index=False,

    encoding="utf-8-sig"

)

# -------------------------------------------------
# 16. Console Result
# -------------------------------------------------

print()
print("=" * 80)

if FINAL_STATUS == "FAIL":

    print("❌ FINAL RESULT: FAIL")

elif FINAL_STATUS == "PASS_WITH_WARNING":

    print("⚠️ FINAL RESULT: PASS WITH WARNING")

else:

    print("✅ FINAL RESULT: PASS")

print("=" * 80)

print()
print(
    f"Passed checks    : {passed}"
)

print(
    f"Warnings         : {warnings}"
)

print(
    f"Critical failures: {critical_failures}"
)

print()
print(
    FINAL_DECISION
)

print()
print(
    f"Saved: {final_result_file}"
)

print("=" * 80)

# -------------------------------------------------
# 17. Final Summary Object
# -------------------------------------------------

PHASE_7_5_STATUS = FINAL_STATUS

print()
print(
    "PHASE 7.5 COMPLETED"
)

print(
    f"STATUS = {PHASE_7_5_STATUS}"
)

print("=" * 80)

Cell 15 — Final PASS / FAIL


,check,status,message
0,Dataset Not Empty,PASS,"81,853 samples detected."
1,Unique Dataset IDs,PASS,0 duplicate IDs.
2,Text Metadata,CRITICAL,"81,853 samples without text."
3,Valid Frame Metadata,PASS,0 invalid frame records.
4,Train Split,PASS,"42,993 training samples."
5,Validation Split,WARNING,0 validation samples.
6,Test Split,PASS,"38,860 test samples."
7,Speaker Diversity,WARNING,1 speaker(s).
8,Video Diversity,WARNING,2 video(s).
9,Duplicate Content,WARNING,"81,853 duplicate-content records."



❌ FINAL RESULT: FAIL

Passed checks    : 5
Warnings         : 4
Critical failures: 1

Dataset is NOT ready for training.

Saved: C:\LipReadingSSL\output\dataset\analysis\final_report\final_result.json

PHASE 7.5 COMPLETED
STATUS = FAIL
